In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from keras import Sequential, Input
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.metrics import Precision, Recall

In [ ]:
df = pd.read_csv('/kaggle/input/titanic/train.csv')
# Basic cleaning
df['Age'] = df['Age'].fillna(df['Age'].median())
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])
df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})
df = pd.get_dummies(df, columns=['Embarked'], drop_first=True)
# Drop irrelevant columns
df = df.drop(columns=['Cabin', 'Name', 'Ticket'])

In [ ]:
# Feature and target
X = df[['Pclass', 'Sex', 'Age', 'Fare', 'SibSp', 'Parch']]
y = df['Survived']

In [ ]:
# Scale the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Print first 5 rows of scaled features
print("Scaled Features (first 5 rows):")
print(X_scaled[:5])

print("Shape of 1 row:")
print(X_scaled[0].shape)

In [ ]:
# Split the data
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)
print("Train data shape:",X_train.shape, "Train Label shape", y_train.shape)
print("Test data shape:",X_test.shape, "Test Label shape", y_test.shape)

In [ ]:
# --- Model 1: Simple 2-layer Network ---
model_1 = Sequential([
    Input(shape=(X_train.shape[1],)),
    layers.Dense(16, activation='relu'),
    layers.Dense(8, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

# --- Model 2: Deeper and with Dropout ---
model_2 = Sequential([
    Input(shape=(X_train.shape[1],)),
    layers.Dense(32, activation='tanh'),
    layers.Dropout(0.3),
    layers.Dense(16, activation='tanh'),
    layers.Dropout(0.3),
    layers.Dense(8, activation='tanh'),
    layers.Dense(1, activation='sigmoid')
])

In [ ]:
# COMPILE BOTH MODELS
for model in [model_1, model_2]:
    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy', Precision(name='precision'), Recall(name='recall')]
    )

In [ ]:
# TRAIN MODEL 1
history_1 = model_1.fit(
    X_train, y_train,
    epochs=50,
    validation_split=0.2,
    verbose=1,
)

In [ ]:
# TRAIN MODEL 2
history_2 = model_2.fit(
    X_train, y_train,
    epochs=50,
    validation_split=0.2,
    verbose=1,
)

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(8, 5))
plt.plot(history_1.history['accuracy'], label='Model 1 Train Acc', color='blue')
plt.plot(history_1.history['val_accuracy'], label='Model 1 Val Acc', linestyle='--', color='blue')
plt.plot(history_2.history['accuracy'], label='Model 2 Train Acc', color='orange')
plt.plot(history_2.history['val_accuracy'], label='Model 2 Val Acc', linestyle='--', color='orange')

plt.title('Training Accuracy Comparison')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
#Evaluation
test_metrics_1 = model_1.evaluate(X_test, y_test, verbose=0)
test_metrics_2 = model_2.evaluate(X_test, y_test, verbose=0)

print("\n--- FINAL TEST RESULTS ---")
print(f"Model 1 -> Accuracy: {test_metrics_1[1]:.4f}, Precision: {test_metrics_1[2]:.4f}, Recall: {test_metrics_1[3]:.4f}")
print(f"Model 2 -> Accuracy: {test_metrics_2[1]:.4f}, Precision: {test_metrics_2[2]:.4f}, Recall: {test_metrics_2[3]:.4f}")

In [ ]:
import shap
import lime
import lime.lime_tabular

In [ ]:
# ---- Choose the better model ----
model = model_1

In [ ]:
# ---- SHAP EXPLANATION ----
# Wrap model prediction for SHAP
def predict_fn(X):
    return model.predict(X)

# Convert arrays back to DataFrame for column names
feature_names = ['Pclass', 'Sex', 'Age', 'Fare', 'SibSp', 'Parch']
X_train_df = pd.DataFrame(X_train, columns=feature_names)
X_test_df = pd.DataFrame(X_test, columns=feature_names)

# Use a small background sample for SHAP
background = shap.sample(X_train_df, 100, random_state=42)

# Let SHAP choose the right explainer (DeepExplainer/KernelExplainer depending on model)
explainer = shap.Explainer(model, background)
shap_values = explainer(X_test_df)

# --- Global SHAP summary plot ---
shap.summary_plot(shap_values.values, X_test_df, feature_names=feature_names, plot_type='bar')

# --- Local SHAP force plot for one prediction ---
i = 0  # index of the test instance
shap.initjs()
shap.plots.force(shap_values[i])  # visualize the impact for one instance

In [ ]:
# ---- LIME EXPLANATION ----

def predict_fn(X):
    preds = model.predict(X)
    return np.concatenate([1 - preds, preds], axis=1)  # shape (n_samples, 2)

explainer_lime = lime.lime_tabular.LimeTabularExplainer(
    X_train,
    feature_names=feature_names,
    class_names=['Did not survive', 'Survived'],
    discretize_continuous=True
)

# Pick a test instance (different from SHAP one)
j = 5
exp = explainer_lime.explain_instance(
    X_test[j],
    predict_fn,
    num_features=6
)

# Show explanation
exp.show_in_notebook()
# Or as a matplotlib plot
# exp.as_pyplot_figure()
